# Domain H: Cell Morphology and Structure–Function Relationships

This notebook addresses two questions using cell-body morphological features from the zarr multimodal data:
- **H1:** Does cell-body morphology predict transcriptomic cell type?
- **H2:** Does soma morphology correlate with functional properties independently of cell type?

**Data:** Zarr multimodal stores containing morphology/mask_properties (n_voxels, size_pc1/2/3_um, angle_deg_xy, centroid positions) and transcriptomic labels.

In [ ]:
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

sns.set_context('talk')
sns.set_style('whitegrid')

# ── Constants ──
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}

# ── Load morphology data from all mice ──
morph_records = []
for mouse_id in MOUSE_IDS:
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    morph = z['morphology/mask_properties']
    subclass_names = z['transcriptomics/cell_type/subclass_name'][:]
    supertype_names = z['transcriptomics/cell_type/supertype_name'][:]
    n_cells = len(subclass_names)
    
    n_voxels = morph['n_voxels'][:]
    size_pc1 = morph['size_pc1_um'][:]
    size_pc2 = morph['size_pc2_um'][:]
    size_pc3 = morph['size_pc3_um'][:]
    angle = morph['angle_deg_xy'][:]
    cx = morph['centroid_x_um'][:]
    cy = morph['centroid_y_um'][:]
    cz = morph['centroid_z_um'][:]
    
    for i in range(n_cells):
        morph_records.append({
            'mouse_id': mouse_id,
            'subclass': subclass_names[i],
            'supertype': supertype_names[i],
            'n_voxels': n_voxels[i],
            'size_pc1_um': size_pc1[i],
            'size_pc2_um': size_pc2[i],
            'size_pc3_um': size_pc3[i],
            'angle_deg_xy': angle[i],
            'centroid_z_um': cz[i],
            'elongation': size_pc1[i] / max(size_pc2[i], 0.1),
            'flatness': size_pc2[i] / max(size_pc3[i], 0.1),
        })

morph_df = pd.DataFrame(morph_records)
morph_df['subclass_short'] = morph_df['subclass'].map(SUBCLASS_SHORT)
present_subclasses = [s for s in SUBCLASS_ORDER if s in morph_df['subclass'].unique()]

print(f"Total cells with morphology: {len(morph_df)}")
print(f"\nMorphology summary:")
print(morph_df[['n_voxels', 'size_pc1_um', 'elongation', 'flatness']].describe().round(2))

## H1: Cell-Body Morphology Predicts Transcriptomic Cell Type

Compare morphological features across subclasses and train a classifier to predict subclass from morphology alone.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H1  Morphological Feature Comparison Across Cell Types
# ══════════════════════════════════════════════════════════════════════

short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

features = ['n_voxels', 'size_pc1_um', 'elongation', 'flatness', 'angle_deg_xy', 'centroid_z_um']
titles = ['Cell Body Volume (voxels)', 'Soma Size PC1 (µm)', 'Elongation (PC1/PC2)',
          'Flatness (PC2/PC3)', 'Soma Orientation Angle (°)', 'Cortical Depth (µm)']

for idx, (feat, title) in enumerate(zip(features, titles)):
    ax = axes[idx // 3, idx % 3]
    sns.violinplot(data=morph_df[morph_df['subclass_short'].isin(short_order)],
                   x='subclass_short', y=feat, order=short_order,
                   palette=short_pal, cut=0, inner='box', ax=ax)
    ax.set_title(f'H1: {title}', fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    
    # Kruskal-Wallis test
    groups = [morph_df.loc[morph_df['subclass'] == s, feat].dropna().values
              for s in present_subclasses if (morph_df['subclass'] == s).sum() >= 3]
    if len(groups) >= 2:
        stat, p = kruskal(*groups)
        ax.text(0.02, 0.98, f'H={stat:.1f}, p={p:.1e}', transform=ax.transAxes,
                fontsize=8, va='top', ha='left',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H1.2  Morphology-Based Cell-Type Classification
# ══════════════════════════════════════════════════════════════════════

# Prepare features
feature_cols = ['n_voxels', 'size_pc1_um', 'size_pc2_um', 'size_pc3_um',
                'elongation', 'flatness', 'angle_deg_xy', 'centroid_z_um']

valid = morph_df[feature_cols + ['subclass_short', 'mouse_id']].dropna()
X = valid[feature_cols].values
y = valid['subclass_short'].values
groups = valid['mouse_id'].values

# Leave-one-mouse-out cross-validation
logo = LeaveOneGroupOut()
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
y_pred = cross_val_predict(clf, X, y, groups=groups, cv=logo)

# Confusion matrix
labels = short_order
cm = confusion_matrix(y, y_pred, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix
ax = axes[0]
sns.heatmap(cm_norm, xticklabels=labels, yticklabels=labels,
            annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('H1: Morphology → Subclass (Leave-Mouse-Out)', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)

# Feature importance
clf.fit(X, y)
importances = clf.feature_importances_
ax = axes[1]
ax.barh(feature_cols, importances, color='steelblue')
ax.set_xlabel('Feature Importance')
ax.set_title('H1: Random Forest Feature Importance', fontweight='bold')

plt.tight_layout()
plt.show()

print(classification_report(y, y_pred, labels=labels, zero_division=0))

# 2D scatter: elongation vs volume
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
for sc in present_subclasses:
    mask = morph_df['subclass'] == sc
    ax.scatter(morph_df.loc[mask, 'n_voxels'], morph_df.loc[mask, 'elongation'],
               c=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc], alpha=0.4, s=15)
ax.set_xlabel('Cell Body Volume (voxels)')
ax.set_ylabel('Elongation (PC1/PC2)')
ax.set_title('H1: Morphological Space by Cell Type', fontweight='bold')
ax.legend(fontsize=8, markerscale=2)
plt.tight_layout()
plt.show()